SETUP

In [4]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print('Libraries are imported successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.4 MB/s eta 0:00:00
Libraries are imported successfully


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

In [5]:
from groq import Groq
API_KEY="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"
client=Groq(api_key=API_KEY)
MODEL="llama-3.1-8b-instant"
print(f"Groq client confirmed with model:{MODEL}")

Groq client confirmed with model:llama-3.1-8b-instant


In [6]:
def ask_llm(user_message,system_message = "You are a helpful assistant.",temperature=0.7,max_tokens=500):
    response=client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

test_response=ask_llm("What is ETL in DataEngineering ? Explain in two lines")
test_response2=ask_llm("Is GenAI & DataEngineering on cloud is good career choice in 2026")
print("=== LLM Response ===")
print(test_response)
print(test_response2)

=== LLM Response ===
In data engineering, ETL (Extract, Transform, Load) is a process used to extract data from various sources, transform it into a standardized format, and load it into a target system, such as a data warehouse or database. ETL helps to integrate, validate, and standardize data to support business intelligence and analytics.
Based on current trends and industry growth, GenAI (Generative AI) and Data Engineering on cloud are excellent career choices in 2026. Here's why:

**GenAI:**

1. **Rapid adoption**: AI and machine learning (ML) are being increasingly adopted across industries, leading to a growing demand for professionals with expertise in GenAI.
2. **New tools and frameworks**: New libraries, frameworks, and tools are being developed to support GenAI, such as PyTorch, TensorFlow, and Hugging Face's Transformers.
3. **Business applications**: GenAI is being applied in various areas, including natural language processing (NLP), computer vision, and predictive anal

In [7]:
response_llm=ask_llm(
    "In 3 bullet points, explain how the Medallion Architecture "
    "(Bronze,Silver,Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineer instructor. "
                    "Be concise and practical"
)
print('Medallion + ETL connection:')
print(response_llm)
print()
print('---Token explanation ---')
print('Each work is roughlt 1-2 tokens.')
print('The model above used approximately',len(response_llm.split())*1.3,'tokens.')
print('llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
Here's how the Medallion Architecture (Bronze, Silver, Gold layers) relates to ETL (Extract, Transform, Load) pipelines:

* **Bronze Layer (Raw Data)**: This layer represents the raw data extracted from various sources (e.g., databases, APIs, files). The ETL pipeline starts with data extraction (Extract) from these sources, which is then loaded into a raw data store (e.g., data lake). The Bronze layer is essentially the input data that needs to be processed further.
* **Silver Layer (Processed Data)**: In this layer, the extracted data is processed (Transform) to clean, aggregate, and enrich the data. The ETL pipeline performs data quality checks, data mapping, and data normalization to create a more refined dataset. The Silver layer stores the processed data in a more structured format (e.g., data warehouse).
* **Gold Layer (Curated Data)**: This layer represents the final, curated data that's been transformed and validated. The ETL pipeline performs any ne

**Prompt Enginerring Experiments**

In [8]:
zero_shot_response=ask_llm(
    "Extract the city name from this address:"
    "456 Brigade Road,Bengaluru,Karnataka,India"
)
print('Zero-Shot REsult:')
print(zero_shot_response)
print()

ambiguous_response=ask_llm("Cleaan this data : Arjun,4500,Mumbai")
print("Ambiguous Result:")
print(ambiguous_response)
print()
print('Problem: Output format is unpredictable and not machine-parsable')

Zero-Shot REsult:
The city name extracted from the address is: Bengaluru.

Ambiguous Result:
It appears you have a simple data point with a name, age (or salary), and location. To clean the data, I'll make an assumption that "4500" represents the salary rather than age, as it's a relatively low number for age.

If you'd like to store this data in a structured format, I can suggest the following cleaned-up version:

- Name: Arjun
- Salary: 4500
- Location: Mumbai

Problem: Output format is unpredictable and not machine-parsable


In [9]:
few_shot_prompt="""
Convert Employee text to JSON,Here is an example;

Imput:Arjun,45000,Mumbai
Output:{"name":"Arjun","salary":45000,"city":"Mumbai"}

Now convert this:
Input:Ananya,38000,Kolkatta
 Output:"""

few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print('Few_shot Result:')
print(few_shot_response)
print()

try:
  parsed=json.loads(few_shot_response.strip())
  print("Successfully parsed as JSON")
  print(f'Name:{parsed["name"]}',
  f'Salary:{parsed["salary"]}',
  f'City:{parsed["city"]}')
except json.JSONDecodeError:
        print('Parsing Failed - model added extra text')
        print('Solution: add explicit instructions in the system prompt')

Few_shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(name, salary, city):
    employee = {
        "name": name,
        "salary": salary,
        "city": city
    }
    return json.dumps(employee)

# Test the function
name = "Ananya"
salary = 38000
city = "Kolkatta"

output = convert_to_json(name, salary, city)
print(output)
```

When you run this code, it will output:

```json
{"name": "Ananya", "salary": 38000, "city": "Kolkatta"}
```

Alternatively, you can use a dictionary to store the employee data and then use the `json.dumps()` function to convert it to a JSON string. This approach is more concise and efficient.

You can also use a function that takes a string as input and splits it into name, salary, and city. Here's an example:

```python
import json

def convert_employee_text_to_json(text):
    name, salary, city = text.split(",")
    return json.dumps({"name": name, "salary": int(salary), "c

In [10]:
prompt="Give me one creative name for a data analytics start up."

print('=== Temperature Experiment ===')
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature {temp} : {response.strip()}')
  time.sleep(1)

print()
print('Observation')
print(' temperature=0.0 -> Same or very similar ans every run(deterministic)')
print(' temperaature=0.5 -> some variation')
print(' temperature=1.0 -> very different')
print()
print('Rule for data Engineering taks: use temperature = 0.0 or  0.1')

=== Temperature Experiment ===
Temperature 0.0 : Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys a sense of innovation and forward-thinking, which is perfect for a data analytics startup.
Temperature 0.5 : Here's a creative name for a data analytics startup:

"Apexion Insights"

"Apexion" is a combination of the words "apex" and "vision," suggesting a company that helps businesses reach new heights through data-driven insights. This name conveys a sense of ambition, innovation, and forward-thinking, which can be appealing to potential clients and partners.
Temperature 1.0 : Here's a creative name for a data analytics start-up:

"Perspectiv.ai"

This name suggests a company that provides insights and perspectives on data, using artificial intelligence to drive business decisions. The ".ai" suffix adds a modern touch, hin

In [11]:
invoice_text = """Invoice #2024-001 from TECHWORLD SOLUTIONS
dated 15th January 2024. Amount: Rs. 45,000 for Laptop"""
# WEAK PROMPT vague, no format specification
weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)
print ('WEAK PROMPT OUTPUT:')
print(weak_response)
print()
try:
  json.loads(weak_response)
  print('PARSEABLE:Yes')
except:
  print('Parseable:No-cannot load into DataFrames')
  print('\n'+ '='*50 + '\n' )

# STRONG PROMPT - role + schema +

strong_system = """You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object.

Do NOT include any explanation, preamble, or markdown formatting.

Return ONLY the JSON, nothing else.
JSON schema (use null for missing values): {"invoice_id": string, "vendor_name": string (Title Case),

"amount": number (no currency symbols),

"currency": string (default INR),

"invoice_date": string (YYYY-MM-DD)

"category": string (Electronics/Se
"""
strong_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    system_message=strong_system,
    temperature=0.3
)
print ('STRONG PROMPT OUTPUT:')
print(strong_response)

try:
  parsed=json.loads(strong_response)
  print('PARSEABLE:Yes')
  print(f'Vendor: {parsed.get("vendor_name")}')
  print(f'Amount: {parsed.get("amount")}')
  print(f'Date: {parsed.get("invoice_date")}')

except json.JSONDecodeError:
  match=re.search(r'JSON schema (.*?)\.',strong_response,re.DOTALL)
  if match:
    parsed=json.loads(match.group())
    print('Parseable:Yes (extracted with regex fallback)')
  else:
    print('PARSABLE : No - retry with stricter prompt')

WEAK PROMPT OUTPUT:
**Invoice Details:**

- **Invoice Number:** 2024-001
- **Invoice Date:** 15th January 2024
- **Invoice Issuer:** TECHWORLD SOLUTIONS
- **Invoice Description:** Laptop
- **Invoice Amount:** Rs. 45,000

**Cleaned and Formatted Invoice:**

Invoice # 2024-001
Date: 15th January 2024
Issued by: TECHWORLD SOLUTIONS
Description: Laptop
Amount: Rs. 45,000.00

Parseable:No-cannot load into DataFrames


STRONG PROMPT OUTPUT:
{"invoice_id": "2024-001", "vendor_name": "Techworld Solutions", "amount": 45000, "currency": "INR", "invoice_date": "2024-01-15", "category": "Electronics/Laptop"}
PARSEABLE:Yes
Vendor: Techworld Solutions
Amount: 45000
Date: 2024-01-15


MINI PROJECT: Smart Data Cleaner

Goal: Convert & messy Involce strings Into a clean, structured Pandas Dataframe using LL.M. This is a complete GenAl-powered ETL pipelinet

Messy Text -> LLM -> 380 -> DataFrame -> Analysis

### Mini-Project: Smart Data Cleaner - GenAI-powered ETL Pipeline

In [12]:
# Sample messy invoice texts
messy_invoices = [
    "Invoice #2024-002 from GLOBEX Corp. on 2024-02-10 for $1200, category: Software licenses.",
    "BILLING: #INV-003 from Acme Services, 2024/03/05. Amount 750 USD. For Consulting.",
    "Receipt no. 4 for 'Digital Marketing Solutions' from WEBMART. Date: March 25, 2024. Total: 3500.00 EUR.",
    "Invoice ID: 2024/005, Vendor: OFFICE SUPPLIES, Date: 12-APR-2024, Total: 150 GBP, for stationary.",
    "TECH SUPPORT INV-006, from IT Solutions, dated April 30, 2024. Amount: 500, CURR: CAD. Category: Service."
]

In [14]:
!pip install groq --quiet
import json
import pandas as pd
from groq import Groq

# API_KEY, client, MODEL, and ask_llm are defined in previous cells (1DjCyS3Cm93M and HkknGen3op8O)
# and should be globally available. No need to redefine them here.

# Define a strong system prompt for invoice extraction
invoice_system_prompt = """You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object.

Do NOT include any explanation, preamble, or markdown formatting.

Return ONLY the JSON, nothing else.
JSON schema (use null for missing values):
{
  "invoice_id": string,
  "vendor_name": string (Title Case),
  "amount": number (no currency symbols),
  "currency": string (default INR),
  "invoice_date": string (YYYY-MM-DD),
  "category": string
}"""

extracted_data = []

for invoice_text in messy_invoices:
    print(f"Processing: {invoice_text[:50]}...")
    try:
        # Use the strong prompt to get structured data
        llm_response = ask_llm(
            user_message=f"Clean this invoice data: {invoice_text}",
            system_message=invoice_system_prompt,
            temperature=0.0 # Use low temperature for deterministic output
        )

        # Attempt to parse the JSON response
        parsed_json = json.loads(llm_response.strip())
        extracted_data.append(parsed_json)
        print("  Successfully extracted and parsed.")
    except json.JSONDecodeError as e:
        print(f"  Error parsing JSON for: {invoice_text[:30]}... Error: {e}")
        print(f"  LLM Response: {llm_response}")
    except Exception as e:
        print(f"  An unexpected error occurred for: {invoice_text[:30]}... Error: {e}")

# Convert the list of dictionaries to a Pandas DataFrame
df_invoices = pd.DataFrame(extracted_data)

print("\n--- Extracted Data DataFrame ---")
display(df_invoices)

Processing: Invoice #2024-002 from GLOBEX Corp. on 2024-02-10 ...
  Successfully extracted and parsed.
Processing: BILLING: #INV-003 from Acme Services, 2024/03/05. ...
  Successfully extracted and parsed.
Processing: Receipt no. 4 for 'Digital Marketing Solutions' fr...
  Successfully extracted and parsed.
Processing: Invoice ID: 2024/005, Vendor: OFFICE SUPPLIES, Dat...
  Successfully extracted and parsed.
Processing: TECH SUPPORT INV-006, from IT Solutions, dated Apr...
  Successfully extracted and parsed.

--- Extracted Data DataFrame ---


,invoice_id,vendor_name,amount,currency,invoice_date,category
0,2024-002,Globex Corp.,1200.0,INR,2024-02-10,Software licenses
1,INV-003,Acme Services,750.0,USD,2024-03-05,Consulting
2,Receipt no. 4,Webmart,3500.0,EUR,2024-03-25,Digital Marketing Solutions
3,2024/005,Office Supplies,150.0,GBP,2024-04-12,stationary
4,INV-006,IT Solutions,500.0,CAD,2024-04-30,Service


Q1: What is the difference between ML. (Day 5) and Generative Al (Day 6)?

Q2: What does temperature-0.0 do in an LLM API call and when would you use it?

Q3: Write a few-shot prompt that extracts name and salary from text in JSON format.

Q4: What is LLM hallucination and how can prompt engineering reduce it?

Q5: Your LLM returnsson\n{"name":"Ramesh")\n*** and json.loads() crashes. Write the fix.

Q6: How does today's Smart Data Cleaner differ from the manual ETL cleaning on Day 37

# Practice Questions – Generative AI

## Q1: What is the difference between Machine Learning (ML) and Generative AI?

**Machine Learning (ML):**
- Learns patterns from data and makes predictions.
- Works mainly with existing data.
- Example: Spam detection, sales prediction.

**Generative AI:**
- Creates new content such as text, images, audio, or code.
- Uses large language models to generate outputs.
- Example: ChatGPT generating answers or writing code.

 ML predicts,  Generative AI creates.

---

## Q2: What does `temperature = 0.0` do in an LLM API call and when would you use it?

The temperature parameter controls the randomness of the model's response.

- **temperature = 0.0**
  - Gives the most consistent and predictable output.
  - Produces the same or very similar answer for the same prompt.
  - Reduces creativity.

**Used for:**
- Data extraction
- JSON generation
- Classification tasks
- Any task where accuracy is more important than creativity.

---

## Q3: Write a few-shot prompt that extracts name and salary from text in JSON format.

```text
Extract name and salary from the text and return only JSON.

Example 1:
Text: Ravi earns ₹50,000 per month.
Output:
{"name":"Ravi","salary":"50000"}

Example 2:
Text: Priya's monthly salary is ₹75,000.
Output:
{"name":"Priya","salary":"75000"}

Now extract:

Text: Ramesh earns ₹60,000 per month.
Output:


## Q4: What is LLM hallucination and how can prompt engineering reduce it?

LLM hallucination happens when a language model generates incorrect or made-up information while sounding confident.

**Example:** Giving a fake fact, reference, or answer that is not actually true.

Prompt engineering can reduce hallucinations by:
- Giving clear and specific instructions.
- Providing enough context.
- Asking the model to use only the given information.
- Requesting structured outputs such as JSON.
- Instructing the model to say "I don't know" when information is unavailable.

**In short:** Hallucination means generating false information, and good prompts help the model stay accurate and focused.

---

## Q5: Your LLM returns:

```text
Sure!

{"name":"Ramesh"}

***

In [15]:
import json
import re

response = '''
Sure!

{"name":"Ramesh"}

***
'''

match = re.search(r'\{.*\}', response, re.DOTALL)

if match:
    data = json.loads(match.group())
    print(data)

{'name': 'Ramesh'}


## Q6: Diff Btw Smart Data Cleaner and ETL Data Cleaning:

| Manual ETL Cleaning               | Smart Data Cleaner                        |
| --------------------------------- | ----------------------------------------- |
| Rules are written manually        | Uses AI to understand and clean data      |
| Requires more coding effort       | Requires less manual coding               |
| Handles predefined cleaning tasks | Can make intelligent cleaning suggestions |
| Fixed workflow                    | More flexible and adaptive                |
| User decides all transformations  | AI assists in detecting and fixing issues |
